# V8.0: Two-Stage Training (BraTS2021 Pretrain + TextBraTS Fine-tune)

**Stage 1:** BraTS2021 (1251 cases, no text) — learn visual features
**Stage 2:** BraTS2020 (295 cases + TextBraTS text) — add text guidance

In [1]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob, threading
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

# Enable Drive auto-sync env var for train.py
os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

# Unzip BraTS2020 + TextBraTS data (for Stage 2)
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'BraTS2020 data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    last = os.path.join(local_ckpt, 'last.pth')
    if os.path.exists(last):
        shutil.copy2(last, os.path.join(DRIVE_CKPT, f'last_{tag}.pth'))
    print(f'Synced to {DRIVE_CKPT}')

# ========== Background Auto-Sync Daemon ==========
_sync_stop = threading.Event()
_sync_track = {}

def _file_is_stable(path, wait=3):
    try:
        s1 = os.path.getsize(path)
        time.sleep(wait)
        s2 = os.path.getsize(path)
        return s1 == s2 and s1 > 0
    except OSError:
        return False

def _bg_sync_loop():
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    while not _sync_stop.is_set():
        _sync_stop.wait(120)
        if _sync_stop.is_set():
            break
        try:
            files = glob.glob(os.path.join(local_ckpt, '*.pth'))
            synced = 0
            for f in files:
                mt = os.path.getmtime(f)
                name = os.path.basename(f)
                if name not in _sync_track or _sync_track[name] < mt:
                    if not _file_is_stable(f):
                        continue
                    shutil.copy2(f, os.path.join(DRIVE_CKPT, name))
                    _sync_track[name] = mt
                    synced += 1
            if synced > 0:
                print(f'[AutoSync] {synced} checkpoint(s) synced to Drive')
        except Exception as e:
            print(f'[AutoSync] Warning: {e}')

_sync_thread = threading.Thread(target=_bg_sync_loop, daemon=True)
_sync_thread.start()
print('Background auto-sync started (every 2 min)')
print('Setup complete')


Mounted at /content/drive
Fri Apr  3 06:07:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   33C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [2]:
# Extract BraTS2021 from Drive to local SSD
import os, subprocess
os.chdir(REPO_DIR)

BRATS2021_ZIP = os.path.join(DRIVE_BASE, 'BraTS2021_archive.zip')
BRATS2021_LOCAL = '/content/BraTS2021'

if os.path.exists(os.path.join(BRATS2021_LOCAL, 'train')):
    cases = os.listdir(os.path.join(BRATS2021_LOCAL, 'train'))
    print(f'BraTS2021 already extracted. Train cases: {len(cases)}')
else:
    assert os.path.exists(BRATS2021_ZIP), f'BraTS2021_archive.zip not found on Drive: {BRATS2021_ZIP}'
    print('Extracting BraTS2021 to local SSD...')
    os.makedirs('/content/brats2021_tmp', exist_ok=True)
    !unzip -q {BRATS2021_ZIP} -d /content/brats2021_tmp/
    !tar xf /content/brats2021_tmp/BraTS2021_Training_Data.tar -C /content/brats2021_tmp/
    # Move extracted cases to final location
    os.makedirs(BRATS2021_LOCAL, exist_ok=True)
    !mv /content/brats2021_tmp/BraTS2021_* {BRATS2021_LOCAL}/ 2>/dev/null; true
    print('Preparing train/val split...')
    !python scripts/prepare_brats.py --input {BRATS2021_LOCAL} --output {BRATS2021_LOCAL}
    # Cleanup
    !rm -rf /content/brats2021_tmp
    cases = os.listdir(os.path.join(BRATS2021_LOCAL, 'train'))
    print(f'Done. Train cases: {len(cases)}')

Extracting BraTS2021 to local SSD...
Preparing train/val split...
Scanning /content/BraTS2021 for cases...
Found 1251 valid cases out of 1251 total

Split: train=875, val=187, test=189

Processing train...
100% 875/875 [00:00<00:00, 6275.84it/s]

Processing val...
100% 187/187 [00:00<00:00, 6600.20it/s]

Processing test...
100% 189/189 [00:00<00:00, 6300.41it/s]

Dataset preparation complete!
Output: /content/BraTS2021
  train: 875 cases
  val: 187 cases
  test: 189 cases
Done. Train cases: 875


## Stage 1: BraTS2021 Pretrain (No Text)

1251 cases, 200 epochs, --no-text-ratio 1.0

In [ ]:
import os, glob, shutil
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

STAGE1_LAST = 'last_stage1.pth'

resume_args = ''
local_ckpt_dir = os.path.join(REPO_DIR, 'checkpoints')
drive_stage1_last = os.path.join(DRIVE_CKPT, STAGE1_LAST)

if os.path.exists(drive_stage1_last):
    # Resume from stage1-specific checkpoint only
    os.makedirs(local_ckpt_dir, exist_ok=True)
    shutil.copy2(drive_stage1_last, os.path.join(local_ckpt_dir, 'last.pth'))
    # Only copy stage1-specific best, not shared best.pth
    s1_best = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
    if os.path.exists(s1_best):
        shutil.copy2(s1_best, os.path.join(local_ckpt_dir, 'best.pth'))
    resume_args = '--resume checkpoints/last.pth'
    print(f'Resuming Stage 1 from: {drive_stage1_last}')
else:
    for f in glob.glob(os.path.join(local_ckpt_dir, '*.pth')):
        os.remove(f)
    print('Starting Stage 1 fresh')

# Background task: also sync last_stage1.pth periodically
import threading, time as _time
_stage1_stop = threading.Event()
def _stage1_sync():
    while not _stage1_stop.is_set():
        _stage1_stop.wait(180)
        if _stage1_stop.is_set():
            break  # every 3 min
        src = os.path.join(local_ckpt_dir, 'last.pth')
        if os.path.exists(src):
            try:
                s1 = os.path.getsize(src)
                _time.sleep(3)
                if os.path.getsize(src) == s1 and s1 > 0:
                    shutil.copy2(src, drive_stage1_last)
            except Exception:
                pass
_t = threading.Thread(target=_stage1_sync, daemon=True)
_t.start()

print('Stage 1: BraTS2021 Pretrain (no text)...')
!python -u train.py \
    --config configs/autoresearch/V8.0_stage1_pretrain.yaml \
    --no-text-ratio 1.0 \
    --grad-accum 1 \
    {resume_args}

# Save stage1-specific checkpoints to Drive
local_last = os.path.join(local_ckpt_dir, 'last.pth')
if os.path.exists(local_last):
    shutil.copy2(local_last, drive_stage1_last)
sync_and_tag('V8.0_stage1')
print('Stage 1 complete!')


In [14]:
# === Check checkpoint epoch + Sync to Drive ===
import torch, shutil, glob, os, subprocess

# 1. Check checkpoint epochs
print('=== Checkpoint Info ===')
for name in ['last_stage1.pth', 'best_V8.0_stage1.pth', 'best_V8.0.pth',
             'best_V9.0.pth', 'best_V9.1.pth', 'last.pth']:
    path = os.path.join(DRIVE_CKPT, name)
    if os.path.exists(path):
        try:
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
            epoch = ckpt.get('epoch', '?')
            best = ckpt.get('best_dice', '?')
            print(f'  {name}: epoch={epoch}, best_dice={best}')
            del ckpt
        except Exception as e:
            sz = os.path.getsize(path) / (1024**2)
            print(f'  {name}: CORRUPT/UNREADABLE ({sz:.0f} MB) - {e}')

# 2. Sync local checkpoints to Drive
print()
local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
files = glob.glob(os.path.join(local_ckpt, '*.pth'))
if files:
    for f in sorted(files):
        n = os.path.basename(f)
        shutil.copy2(f, os.path.join(DRIVE_CKPT, n))
    print(f'Synced {len(files)} local checkpoint(s) to Drive')
else:
    print('No local checkpoints to sync')

# 3. List Drive checkpoints
print()
print('=== Drive Checkpoints ===')
for f in sorted(glob.glob(os.path.join(DRIVE_CKPT, '*.pth'))):
    sz = os.path.getsize(f) / (1024**2)
    from datetime import datetime
    ts = datetime.fromtimestamp(os.path.getmtime(f)).strftime('%m-%d %H:%M')
    print(f'  {os.path.basename(f)}: {sz:.0f} MB ({ts})')


## Stage 2: BraTS2020 + TextBraTS Fine-tune

Resume from Stage 1, reset optimizer, add text guidance.

In [17]:
# Stop Stage 1 sync thread to prevent overwriting last_stage1.pth
try:
    _stage1_stop.set()  # signal Stage 1 sync thread to stop
except Exception:
    pass

import os, glob
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Only accept versioned Stage 1 checkpoint — no fallback to shared names
STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
assert os.path.exists(STAGE1_CKPT), (
    f'Stage 1 checkpoint not found: {STAGE1_CKPT}\n'
    f'Run Stage 1 first, or check Drive checkpoints.'
)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

print(f'Stage 2: Fine-tune from {STAGE1_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V8.0_stage2_finetune.yaml \
    --resume "{STAGE1_CKPT}" \
    --reset-optimizer \
    --reset-lr \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V8.0')
print('Stage 2 complete!')


## Evaluation

In [18]:
import subprocess, re, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V8.0.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

CONFIG = 'configs/autoresearch/V8.0_stage2_finetune.yaml'
BASELINE = {'ET': 0.7910, 'TC': 0.8560, 'WT': 0.8967, 'Mean': 0.8479}

for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    for line in ret.stdout.split('\n'):
        if 'dice_' in line or 'hd95_' in line:
            print(f'  {line.strip()}')
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-300:]}')

print()
print(f'Baseline V5.0: ET={BASELINE["ET"]}, TC={BASELINE["TC"]}, WT={BASELINE["WT"]}, Mean={BASELINE["Mean"]}')
print('Target: ET > 0.833 (TextBraTS SOTA)')

Using: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V8.0.pth

text+TTA
  dice_ET: 0.8116 +/- 0.2028
  dice_TC: 0.8977 +/- 0.0807
  dice_WT: 0.9167 +/- 0.0511
  dice_mean: 0.8753 +/- 0.0821
  hd95_ET: 2.29 +/- 5.96
  hd95_TC: 1.48 +/- 2.57
  hd95_WT: 1.16 +/- 1.46

notext+TTA
  dice_ET: 0.8115 +/- 0.2027
  dice_TC: 0.8977 +/- 0.0806
  dice_WT: 0.9167 +/- 0.0511
  dice_mean: 0.8753 +/- 0.0821
  hd95_ET: 2.30 +/- 5.98
  hd95_TC: 1.48 +/- 2.56
  hd95_WT: 1.16 +/- 1.46

Baseline V5.0: ET=0.791, TC=0.856, WT=0.8967, Mean=0.8479
Target: ET > 0.833 (TextBraTS SOTA)


## V9.0: Stage 2 with Fusion Fix

4 improvements over V8.0 Stage 2:
1. Freeze vision backbone for first 10 epochs (text warmup)
2. Non-zero fusion init (already in code)
3. Vision modality dropout 15% (force text usage)
4. InfoNCE alignment loss (alignment_weight=0.1)


In [3]:
import os, glob
os.chdir(REPO_DIR)

os.environ['DRIVE_CKPT_DIR'] = DRIVE_CKPT

# Only accept versioned Stage 1 checkpoint
STAGE1_CKPT = os.path.join(DRIVE_CKPT, 'best_V8.0_stage1.pth')
assert os.path.exists(STAGE1_CKPT), f'Not found: {STAGE1_CKPT}'

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)

print(f'V9.0 Stage 2: Fine-tune from {STAGE1_CKPT}')
!python -u train.py \
    --config configs/autoresearch/V9.0_freeze_warmup.yaml \
    --resume "{STAGE1_CKPT}" \
    --reset-optimizer \
    --no-text-ratio 0.15 \
    --freeze-vision-epochs 10 \
    --vision-dropout 0.15 \
    --grad-accum 2

sync_and_tag('V9.0')
print('V9.0 Stage 2 complete!')


In [4]:
import subprocess, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V9.0.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
assert os.path.exists(ckpt), f'No V9.0 checkpoint: {ckpt}'
print(f'Evaluating: {ckpt}')

CONFIG = 'configs/autoresearch/V9.0_freeze_warmup.yaml'
BASELINE = {'V8.0_text': 0.8753, 'V8.0_notext': 0.8753, 'V5.0': 0.8479}

results = {}
for name, flags in [('text+TTA', ['--use-text', '--tta']), ('notext+TTA', ['--no-text', '--tta'])]:
    print()
    print('=' * 60)
    print(name)
    print('=' * 60)
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test', '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)
    for line in ret.stdout.split('\\n'):
        if 'dice_' in line or 'hd95_' in line:
            print(f'  {line.strip()}')
    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-300:]}')

print()
print('Comparison:')
print(f'  V8.0: Mean={BASELINE["V8.0_text"]}, text_delta=0.00%')
print(f'  V5.0: Mean={BASELINE["V5.0"]}, text_delta=+0.55%')
print(f'  TextBraTS SOTA: text_delta=+1.5%')


Evaluating: /content/drive/MyDrive/TextMamba3D/checkpoints/best_V9.0.pth

text+TTA
  Loaded checkpoint: epoch=4, best_dice=0.90061973200904
TextBraTS test: 95 samples

Evaluating 95 cases (test split)
Sliding window: patch=(128, 128, 128), overlap=0.5, text=True
TTA: 8-fold flip ensemble ENABLED

  BraTS20_Training_328: Dice=0.6949 (ET=0.3528, TC=0.7989, WT=0.9328) HD95_ET=50.84
  BraTS20_Training_028: Dice=0.7832 (ET=0.6377, TC=0.8666, WT=0.8452) HD95_ET=1.73
  BraTS20_Training_289: Dice=0.6012 (ET=0.0000, TC=0.8552, WT=0.9484) HD95_ET=nan
  BraTS20_Training_231: Dice=0.9435 (ET=0.9254, TC=0.9427, WT=0.9624) HD95_ET=1.00
  BraTS20_Training_261: Dice=0.7427 (ET=0.5389, TC=0.7372, WT=0.9522) HD95_ET=5.10
  BraTS20_Training_163: Dice=0.9219 (ET=0.8520, TC=0.9396, WT=0.9742) HD95_ET=1.00
  BraTS20_Training_345: Dice=0.8581 (ET=0.8451, TC=0.8295, WT=0.8996) HD95_ET=6.78
  BraTS20_Training_139: Dice=0.9017 (ET=0.8979, TC=0.9282, WT=0.8791) HD95_ET=1.00
  BraTS20_Training_063: Dice=0.7934 (E